In [0]:
# Updated hr_gold.py
# Changes: Used the latest version from your provided code (with rounding to whole numbers).
# Dropped ingestion columns early in the helper function to avoid carrying them.
# This should work in streaming as the silver target is incrementally updated, and gold will recompute on each trigger (suitable for small datasets).
# For large data, consider optimizing further, but assuming HR data is small.

# Databricks notebook source
import dlt
import pyspark.sql.functions as F

# ---------- Helper: build KPIs for a single department & prefix columns (rounded to whole numbers) ----------
def build_prefixed_kpis_for_dept_whole(df, dept_name: str, prefix: str):
    """
    Filter to a given department, compute KPIs, and return a single-row DF
    with columns prefixed (e.g., hr_total_employees, hr_attrition_rate, ...),
    with numeric KPI values rounded to whole numbers.
    """
    dept_df = df.filter(F.col("Department") == dept_name)

    # Drop ingestion metadata columns from the detail rows
    INGESTION_COLS = [
        "input_file_path", "input_file_name", "ingest_timestamp",
        "inputfilepath", "inputfilename", "ingestiontime"
    ]
    dept_df_clean = dept_df.drop(*[c for c in INGESTION_COLS if c in dept_df.columns])

    # Base aggregates
    total_employees = F.count("*")
    attritions = F.sum(F.col("is_attrited").cast("int"))
    safe_denominator = F.when(F.count("*") > 0, F.count("*")).otherwise(F.lit(1))
    attrition_rate_fraction = attritions / safe_denominator  # 0..1

    avg_monthly_income = F.avg(F.col("MonthlyIncome"))
    avg_annual_income = F.avg(F.col("AnnualIncome"))
    avg_years_at_company = F.avg(F.col("YearsAtCompany"))
    avg_age = F.avg(F.col("Age"))

    # Round all KPI outputs to whole numbers (nearest integer)
    kpis = dept_df.agg(
        total_employees.alias(f"{prefix}_total_employees"),
        F.round(attrition_rate_fraction).cast("int").alias(f"{prefix}_attrition_rate"),
        F.round(avg_monthly_income).cast("int").alias(f"{prefix}_avg_monthly_income"),
        F.round(avg_annual_income).cast("int").alias(f"{prefix}_avg_annual_income"),
        F.round(avg_years_at_company).cast("int").alias(f"{prefix}_avg_years_at_company"),
        F.round(avg_age).cast("int").alias(f"{prefix}_avg_age"),
    ).withColumn(f"{prefix}_department", F.lit(dept_name))

    return dept_df_clean, kpis

# =========================
# GOLD: Human Resources
# =========================
@dlt.table(
    name="gold_hr_detail_metrics",
    comment="Human Resources detail rows with HR-only KPIs (rounded whole numbers, prefixed as hr_*), ingestion columns dropped",
    table_properties={"quality": "gold"}
)
def gold_hr_detail_metrics():
    silver = dlt.read("silver_hr_analytics")
    hr_detail, hr_kpis = build_prefixed_kpis_for_dept_whole(silver, "Human Resources", "hr")

    # Cross join single-row KPIs with HR detail rows → every HR row carries HR KPIs
    enriched_hr = hr_detail.crossJoin(hr_kpis)

    return enriched_hr

# =========================
# GOLD: Sales
# =========================
@dlt.table(
    name="gold_sales_detail_metrics",
    comment="Sales detail rows with Sales-only KPIs (rounded whole numbers, prefixed as sales_*), ingestion columns dropped",
    table_properties={"quality": "gold"}
)
def gold_sales_detail_metrics():
    silver = dlt.read("silver_hr_analytics")
    sales_detail, sales_kpis = build_prefixed_kpis_for_dept_whole(silver, "Sales", "sales")

    enriched_sales = sales_detail.crossJoin(sales_kpis)
    return enriched_sales

# =========================
# GOLD: Research & Development
# =========================
@dlt.table(
    name="gold_rd_detail_metrics",
    comment="R&D detail rows with R&D-only KPIs (rounded whole numbers, prefixed as rd_*), ingestion columns dropped",
    table_properties={"quality": "gold"}
)
def gold_rd_detail_metrics():
    silver = dlt.read("silver_hr_analytics")
    rd_detail, rd_kpis = build_prefixed_kpis_for_dept_whole(silver, "Research & Development", "rd")

    enriched_rd = rd_detail.crossJoin(rd_kpis)
    return enriched_rd
     

In [0]:
# =========================
# GOLD: Complete Clean Employee Dataset (All Departments Combined)
# =========================
@dlt.table(
    name="gold_hr_all_employees",
    comment="Complete, clean, deduplicated HR dataset - ALL employees from ALL departments in ONE table (perfect for dashboards & reporting)",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.enabled": "true",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def gold_hr_all_employees():
    """
    Returns the full clean Silver table with:
    - All employees (all departments)
    - All cleaned & derived columns
    - Ingestion metadata dropped (clean look)
    - Ready for Power BI, Tableau, SQL dashboards, etc.
    """
    df = dlt.read("silver_hr_analytics")

    # Drop internal ingestion columns - keep only business + derived columns
    clean_df = df.drop(
        "input_file_path",
        "input_file_name", 
        "ingest_timestamp"
    )

    return clean_df
     